# Graph-EFM Temporal Shift

Probabilistic calibration transfer under temporal distribution shift.

Clones code from GitHub. Stores data, checkpoints, and outputs on Google Drive.


In [ ]:
import os, sys, subprocess, shutil, time

# ---- Configure (repo URL pre-filled; set RUN_PROFILE if needed) ----
GITHUB_REPO = "https://github.com/hugoaslm/leap-graph-efm-shift.git"
NEURAL_LAM_REPO = "https://github.com/mllam/neural-lam.git"
NEURAL_LAM_BRANCH = "prob_model_global"

RUN_PROFILE = "l4_core"
DO_PREPROCESS = True
DO_TRAIN = True
DO_EVALUATE = True
DO_PLOT = True

# ---- Mount Drive ----
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/leap_project"
PROJECT_DIR = "/content/leap_project"
REPO_DIR = os.path.join(PROJECT_DIR, "leap-graph-efm-shift")
NLAM_DIR = os.path.join(PROJECT_DIR, "neural-lam-prob-model")
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(PROJECT_DIR, exist_ok=True)

# ---- Clone repos ----
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO} {REPO_DIR} 2>&1 | tail -3
else:
    %cd {REPO_DIR}
    !git pull 2>&1 | tail -3
    %cd {PROJECT_DIR}

if not os.path.exists(NLAM_DIR):
    !git clone --branch {NEURAL_LAM_BRANCH} {NEURAL_LAM_REPO} {NLAM_DIR} 2>&1 | tail -3

# ---- Apply patches and verify ----
patches_src = os.path.join(REPO_DIR, "neural-lam-patches")
if os.path.exists(patches_src):
    for root, dirs, files in os.walk(patches_src):
        rel = os.path.relpath(root, patches_src)
        for f in files:
            src = os.path.join(root, f)
            dst = os.path.join(NLAM_DIR, rel, f)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(src, dst)
    # Verify the key patch took effect
    ct_path = os.path.join(NLAM_DIR, "neural_lam", "constants.py")
    with open(ct_path) as fh:
        if "load_experiment_config" not in fh.read():
            raise RuntimeError("constants.py patch missing — did you push neural-lam-patches/ ?")
    print("Patches verified.")
else:
    raise RuntimeError(f"neural-lam-patches/ not found at {patches_src}")

# ---- Ensure output dirs ----
for d in ["checkpoints", "results", "figures", "data"]:
    os.makedirs(os.path.join(PROJECT_DIR, d), exist_ok=True)

sys.path.insert(0, NLAM_DIR)
print("Setup done.")


In [ ]:
!pip install -q "numpy>=1.24.2,<2.0.0" scipy matplotlib "xarray>=2024.2.0" "zarr>=2.17.1,<3" gcsfs dask wandb pyyaml tqdm
!pip install -q pytorch-lightning torch-geometric==2.3.1
!pip install -q shapely networkx Cartopy pyproj tueplots
!pip install -q git+https://github.com/deepmind/graphcast.git 2>&1 | tail -2
!pip install -q parse dataclass-wizard
print("Dependencies installed.")


In [ ]:
import importlib, yaml
from neural_lam import constants
importlib.reload(constants)  # ensure patches are loaded

CONFIG_PATH = os.path.join(REPO_DIR, "configs", "wb2_shift_64x32_graph_efm.yaml")
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

constants.load_experiment_config(CONFIG_PATH)

data_name = cfg["dataset"]["name"]
drive_data = os.path.join(DRIVE_ROOT, "data", data_name)
local_data = os.path.join(PROJECT_DIR, "data", data_name)
nl_data = os.path.join(NLAM_DIR, "data", data_name)
os.makedirs(local_data, exist_ok=True)
os.makedirs(nl_data, exist_ok=True)

print(f"Grid: {constants.GRID_SHAPE},  vars: {constants.PARAM_NAMES_SHORT}")
print(f'Splits: train {cfg["splits"]["train"]}, val {cfg["splits"]["val"]}')
print(f'        id {cfg["splits"]["id"]}, ood {cfg["splits"]["ood"]}')


## Data preparation

Downloads WeatherBench 2 ERA5 64x32 if not cached on Drive,
then runs preprocessing (forcing, grid features, mesh, statistics).


In [ ]:
import xarray as xr

required_fields = {
    "geopotential", "temperature", "2m_temperature",
    "geopotential_at_surface", "land_sea_mask",
}

def valid_fields_store(path):
    if not os.path.exists(path):
        return False
    try:
        ds = xr.open_zarr(path, consolidated=False)
    except Exception as exc:
        print(f"Invalid fields cache at {path}: {exc}")
        return False
    missing = required_fields - set(ds.data_vars)
    expected_grid = tuple(cfg["grid"]["shape"])
    actual_grid = (ds.sizes.get("longitude"), ds.sizes.get("latitude"))
    state_dims = ds["2m_temperature"].dims if "2m_temperature" in ds else ()
    static_dims = ds["land_sea_mask"].dims if "land_sea_mask" in ds else ()
    ordered = state_dims == ("time", "longitude", "latitude") and static_dims == ("longitude", "latitude")
    if missing or actual_grid != expected_grid or not ordered:
        print(f"Invalid fields cache at {path}: missing={sorted(missing)}, grid={actual_grid}, dims={state_dims}")
        return False
    return True

def retire_invalid_store(path):
    if os.path.lexists(path) and not valid_fields_store(path):
        backup = f"{path}.invalid_{time.strftime('%Y%m%d_%H%M%S')}"
        print(f"Moving invalid cache aside: {path} -> {backup}")
        os.rename(path, backup)

if not DO_PREPROCESS:
    print("Skipping preprocessing.")
else:
    fields_zarr = os.path.join(local_data, "fields.zarr")
    drive_fields_zarr = os.path.join(drive_data, "fields.zarr")
    nl_fields_zarr = os.path.join(nl_data, "fields.zarr")
    retire_invalid_store(fields_zarr)
    retire_invalid_store(drive_fields_zarr)
    retire_invalid_store(nl_fields_zarr)
    if not valid_fields_store(fields_zarr):
        if valid_fields_store(drive_fields_zarr):
            print("Copying data from Drive cache...")
            shutil.copytree(drive_data, local_data, dirs_exist_ok=True)
        else:
            print("Downloading WB2 data (10-30 min, ~4 GB on disk; requires Google auth popup)...")
            subprocess.run([sys.executable,
                f"{REPO_DIR}/scripts/download_wb2_data.py",
                "--output", fields_zarr,
                '--time_start', cfg['splits']['train'][0],
                '--time_end', cfg['splits']['ood'][1],
                "--method", "xarray", "--colab_auth"], check=True)
            drive_ds = os.path.join(DRIVE_ROOT, "data", data_name)
            os.makedirs(drive_ds, exist_ok=True)
            shutil.copytree(fields_zarr, drive_fields_zarr, dirs_exist_ok=True)
    else:
        print("Data already on local disk.")

    if not valid_fields_store(fields_zarr):
        raise RuntimeError(f"Downloaded fields store failed validation: {fields_zarr}")
    if not os.path.lexists(nl_fields_zarr):
        os.symlink(fields_zarr, nl_fields_zarr)
    if not valid_fields_store(nl_fields_zarr):
        raise RuntimeError(f"neural-lam cannot read fields store: {nl_fields_zarr}")

    for step in ["forcing", "grid_features", "mesh", "parameter_weights"]:
        print(f"Running preprocessing step: {step}")
        subprocess.run([sys.executable, "-u",
            f"{REPO_DIR}/scripts/prepare_wb2_subset.py",
            "--config", CONFIG_PATH, "--steps", step],
            cwd=NLAM_DIR, check=True)


## CPU smoke check

Instantiates Graph-EFM with random weights and verifies shapes.


In [ ]:
import torch
from neural_lam.models.graph_efm import GraphEFM
from types import SimpleNamespace

mc, fc, gc, ev = cfg["model"], cfg["forecast"], cfg["graph"], cfg["evaluation"]
args = SimpleNamespace(
    hidden_dim=mc["hidden_dim"], latent_dim=mc["latent_dim"],
    hidden_layers=mc["hidden_layers"],
    processor_layers=mc["decoder_processor_layers"],
    encoder_processor_layers=mc["encoder_processor_layers"],
    prior_processor_layers=mc["prior_processor_layers"],
    kl_beta=mc["kl_beta"], crps_weight=mc["crps_weight"],
    loss=mc["loss"], sample_obs_noise=mc["sample_obs_noise"],
    output_std=mc["output_std"], prior_dist=mc["prior_dist"],
    learn_prior=mc["learn_prior"],
    graph=gc["name"], dataset=cfg["dataset"]["name"],
    lr=cfg["training"]["phase_a"]["lr"],
    step_length=cfg["sampling"]["step_length"],
    eval_leads=fc["eval_leads"], ensemble_size=ev["ensemble_size"],
    n_example_pred=0, batch_size=1,
)

model = GraphEFM(args).eval()
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

B, M, T = 1, 4, fc["eval_leads"]
N = constants.GRID_SHAPE[0] * constants.GRID_SHAPE[1]
D, F = constants.GRID_STATE_DIM, constants.GRID_FORCING_DIM

with torch.no_grad():
    traj, _ = model.sample_trajectories(
        torch.randn(B, 2, N, D), torch.randn(B, T, N, F),
        torch.randn(B, T, N, D), M)
assert traj.shape == (B, M, T, N, D)
print(f"Trajectory shape {traj.shape} - OK")


## GPU setup & memory benchmark


In [ ]:
assert torch.cuda.is_available(), "No GPU - change runtime type."

gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_mem / 1024**3
print(f"GPU: {gpu}  ({vram:.1f} GB)")

model = model.to("cuda")
B, T = cfg["training"]["phase_a"]["batch_size"], fc["eval_leads"]
N = constants.GRID_SHAPE[0] * constants.GRID_SHAPE[1]
D, F = constants.GRID_STATE_DIM, constants.GRID_FORCING_DIM

init = torch.randn(B, 2, N, D, device="cuda")
targ = torch.randn(B, T, N, D, device="cuda")
forc = torch.randn(B, T, N, F, device="cuda")

torch.cuda.reset_peak_memory_stats()
with torch.no_grad():
    _ = model.sample_trajectories(init, forc, targ, 2)
peak = torch.cuda.max_memory_allocated() / 1024**2
est = peak * 4
print(f"Memory: {peak:.0f} MB (ens=2),  ~{est:.0f} MB (ens=8)")
if est > vram * 1024 * 0.85:
    print("WARNING: may OOM, reduce batch_size.")


## Training - Phase A (ar_steps=1, 100 epochs)

Checkpoints saved to Drive so training survives disconnects.


In [ ]:
if not DO_TRAIN:
    print("Skipping training.")
else:
    ckpt = os.path.join(PROJECT_DIR, "checkpoints", "best_phase_a.ckpt")
    if not os.path.exists(ckpt):
        drive_ckpt = os.path.join(DRIVE_ROOT, "checkpoints", "best_phase_a.ckpt")
        if os.path.exists(drive_ckpt):
            os.makedirs(os.path.dirname(ckpt), exist_ok=True)
            shutil.copy2(drive_ckpt, ckpt)
            print("Restored Phase A checkpoint from Drive.")
    if os.path.exists(ckpt):
        print("Phase A checkpoint exists.")
    else:
        %cd {PROJECT_DIR}
        !python {REPO_DIR}/scripts/train_shift_model.py \
            --config {CONFIG_PATH} --phase a --max_minutes 120
        if os.path.exists(ckpt):
            drive_ckpt = os.path.join(DRIVE_ROOT, "checkpoints")
            os.makedirs(drive_ckpt, exist_ok=True)
            shutil.copy2(ckpt, os.path.join(drive_ckpt, "best_phase_a.ckpt"))
            print("Saved to Drive.")

## Training - Phase B (ar_steps=4, 50 epochs)

Resumes from the Phase A checkpoint.


In [ ]:
if DO_TRAIN:
    ckpt_a = os.path.join(PROJECT_DIR, "checkpoints", "best_phase_a.ckpt")
    ckpt_b = os.path.join(PROJECT_DIR, "checkpoints", "best_phase_b.ckpt")
    if not os.path.exists(ckpt_a):
        ckpt_a = os.path.join(DRIVE_ROOT, "checkpoints", "best_phase_a.ckpt")
        if os.path.exists(ckpt_a):
            shutil.copy2(ckpt_a, os.path.join(PROJECT_DIR, "checkpoints", "best_phase_a.ckpt"))
    if os.path.exists(ckpt_b):
        print("Phase B checkpoint exists.")
    elif not os.path.exists(ckpt_a):
        print("ERROR: Phase A checkpoint not found.")
    else:
        %cd {PROJECT_DIR}
        !python {REPO_DIR}/scripts/train_shift_model.py \
            --config {CONFIG_PATH} --phase b --resume {ckpt_a} --max_minutes 120
        if os.path.exists(ckpt_b):
            drive_ckpt = os.path.join(DRIVE_ROOT, "checkpoints")
            os.makedirs(drive_ckpt, exist_ok=True)
            shutil.copy2(ckpt_b, os.path.join(drive_ckpt, "best_phase_b.ckpt"))


## Evaluation

Ensemble forecasts on validation, ID (2011-2015), and OOD (2016-2022).
Fits post-hoc spread calibration on validation, computes bootstrap CIs.


In [ ]:
if not DO_EVALUATE:
    print("Skipping evaluation.")
else:
    ckpt_b = os.path.join(PROJECT_DIR, "checkpoints", "best_phase_b.ckpt")
    if not os.path.exists(ckpt_b):
        ckpt_b = os.path.join(DRIVE_ROOT, "checkpoints", "best_phase_b.ckpt")
    if not os.path.exists(ckpt_b):
        raise RuntimeError(f"Checkpoint not found: {ckpt_b}")

    results_dir = os.path.join(PROJECT_DIR, "results")
    os.makedirs(results_dir, exist_ok=True)

    %cd {PROJECT_DIR}
    !python {REPO_DIR}/scripts/evaluate_shift.py \
        --config {CONFIG_PATH} --checkpoint {ckpt_b} \
        --ensemble_size {cfg['evaluation']['ensemble_size']} \
        --output {results_dir} --device cuda

    drive_res = os.path.join(DRIVE_ROOT, "results")
    if os.path.exists(drive_res):
        shutil.rmtree(drive_res)
    shutil.copytree(results_dir, drive_res)

    import csv
    mp = os.path.join(results_dir, "metrics.csv")
    if os.path.exists(mp):
        with open(mp) as f:
            rows = list(csv.DictReader(f))
        for r in rows:
            if (r["split"] in ("id","ood") and r["calibration"]=="raw"
                    and r["variable"]=="t2m" and r["metric"]=="rmse"
                    and int(float(r["lead_hours"]))==24):
                print(f"  {r['split']} t2m RMSE@24h = {float(r['mean']):.3f}")
    print("Evaluation done.")


## Figures


In [ ]:
if not DO_PLOT:
    print("Skipping plots.")
else:
    figs_dir = os.path.join(PROJECT_DIR, "figures")
    os.makedirs(figs_dir, exist_ok=True)

    %cd {PROJECT_DIR}
    !python {REPO_DIR}/scripts/plot_shift_results.py \
        --config {CONFIG_PATH} \
        --metrics {os.path.join(PROJECT_DIR, 'results')} \
        --output {figs_dir}

    from IPython.display import Image as IPImg, display
    for fn in sorted(os.listdir(figs_dir)):
        if fn.endswith(".pdf"):
            display(IPImg(filename=os.path.join(figs_dir, fn)))

    drive_figs = os.path.join(DRIVE_ROOT, "figures")
    if os.path.exists(drive_figs):
        shutil.rmtree(drive_figs)
    shutil.copytree(figs_dir, drive_figs)


## Done

Outputs on Google Drive under `MyDrive/leap_project/`:

- `checkpoints/best_phase_b.ckpt` - trained model
- `results/metrics.csv` - per-variable, per-lead, per-split metrics
- `results/calibration_multipliers.json` - fitted spread scaling
- `results/bootstrap_ci.csv` - OOD-ID confidence intervals
- `figures/01-05_*.pdf` - figures

### Troubleshooting

- Non-finite losses: check data normalization, reduce learning rate.
- OOM: reduce `batch_size` in the config YAML.
- Near-zero spread: increase `kl_beta`, check prior network.
- Disconnect: re-run; training resumes from latest Drive checkpoint.
